# Medical Chatbot Fine-Tuning - Data Preprocessing Pipeline

Quy trình này chuẩn bị dữ liệu y tế Việt Nam để fine-tune mô hình ngôn ngữ Qwen2 7B Instruct.

## Các bước chính:
1. Cài đặt thư viện - Cài Apache Arrow và Fastparquet
2. Import thư viện - Tải các thư viện cần thiết
3. Load dữ liệu - Tải dataset từ Hugging Face
4. Khám phá dữ liệu - Xem shape, head, thống kê cơ bản
5. Phân tích chất lượng - Kiểm tra missing values, độ dài
6. Làm sạch dữ liệu - Xóa trùng lặp, lọc độ dài
7. Tokenization - Chuyển text thành token IDs
8. Kiểm tra kết quả - Verify dữ liệu đã tokenize
9. Phân tích token - Xem thống kê độ dài token
10. Chia tập và lưu - Train/validation split và lưu dữ liệu

## Bước 1: Cài đặt thư viện

Cài đặt các thư viện cần thiết:
- pyarrow: Đọc định dạng Parquet
- fastparquet: Hỗ trợ thêm cho Parquet
- datasets: Làm việc với dataset Hugging Face

In [1]:
#!pip install pyarrow fastparquet datasets

## Bước 2: Import thư viện chính

Import các thư viện cần thiết:
- pandas & numpy: Xử lý dữ liệu tabular
- torch: Deep learning framework
- AutoTokenizer: Tải tokenizer từ Hugging Face
- Dataset: Tạo dataset Hugging Face

In [2]:
import re
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer
from datasets import Dataset

## Bước 3: Load dữ liệu từ Hugging Face

Tải dataset y tế Việt Nam từ Hugging Face Hub.
Dataset này đã được cleaned và có sẵn word count cho câu hỏi và trả lời.

In [3]:
print("Loading data...")

data = pd.read_parquet("hf://datasets/ynguyen1010/medical_vietnamese_datasets/cleaned_format/train-00000-of-00001.parquet")

print("Preprocessing data...")

Loading data...
Preprocessing data...


## Bước 4: Khám phá dữ liệu

Kiểm tra shape (kích thước dataset) và xem mẫu dữ liệu đầu tiên.

In [4]:
data.shape

(68498, 4)

In [5]:
data.head()

,question_cleaned,answer_cleaned,q_word_count,a_word_count
0,Căng cơ đùi: Nguyên nhân và các phương pháp đi...,Căng cơ đùi có thể coi là một trong những chấn...,11,1211
1,Chuột rút cơ lưng: Nguyên nhân và biện pháp ph...,Chuột rút cơ lưng có thể gây đau đớn và ảnh hư...,11,1506
2,Gãy xương đòn bao lâu thì tháo nẹp và những lư...,Gãy xương đòn bao lâu thì tháo nẹp là câu hỏi ...,14,1199
3,Cách chữa viêm khớp cùng chậu tại nhà: Phương ...,Cách chữa viêm khớp cùng chậu tại nhà như bài ...,13,1548
4,Gãy xương cẳng tay bao lâu thì lành và những đ...,Gãy xương cẳng tay bao lâu thì lành là điều mà...,14,921


## Bước 5: Phân tích chất lượng dữ liệu

Kiểm tra:
- Missing values: Câu hỏi/trả lời bị thiếu
- Thống kê độ dài: Min, max, mean của word count
- Mẫu quá ngắn: Đếm câu hỏi < 3 từ và trả lời < 10 từ (sẽ xóa)

In [6]:
print("=" * 55)
print("KIỂM TRA DỮ LIỆU ĐẦU VÀO")
print("=" * 55)

# Missing values
missing = data[['question_cleaned', 'answer_cleaned']].isnull().sum()
print(f"\n[Missing values]")
print(missing)

# Thống kê độ dài
print(f"\n[Thống kê độ dài (word count)]")
stats = data[['q_word_count', 'a_word_count']].describe().round(1)
print(stats)

# Số mẫu cực ngắn (cần lọc)
short_q = (data['q_word_count'] < 3).sum()
short_a = (data['a_word_count'] < 10).sum()
print(f"\n Câu hỏi < 3 từ  : {short_q:,} mẫu")
print(f"\n Câu trả lời < 10 từ: {short_a:,} mẫu")

KIỂM TRA DỮ LIỆU ĐẦU VÀO

[Missing values]
question_cleaned    0
answer_cleaned      0
dtype: int64

[Thống kê độ dài (word count)]
       q_word_count  a_word_count
count       68498.0       68498.0
mean           26.2        1154.0
std            33.7         682.0
min             2.0          12.0
25%            10.0         696.0
50%            13.0        1238.0
75%            18.0        1558.0
max           565.0       10716.0

 Câu hỏi < 3 từ  : 3 mẫu

 Câu trả lời < 10 từ: 0 mẫu


## Bước 6: Làm sạch dữ liệu

Tiến hành:
1. Xem chi tiết câu hỏi quá ngắn
2. Kiểm tra và xóa các cặp Q-A trùng lặp
3. Loại bỏ câu hỏi quá ngắn
4. Lọc câu trả lời theo độ dài hợp lý (50-2000 từ)

In [7]:
data[data.duplicated(subset=['question_cleaned', 'answer_cleaned'], keep=False)]

,question_cleaned,answer_cleaned,q_word_count,a_word_count
55581,Tôi bị tai nạn giao thông vùng đầu vào ngày 30...,bác sĩ xin giải đáp như sau: Chóng mặt có rất ...,143,184
55875,Tôi bị tai nạn giao thông vùng đầu vào ngày 30...,bác sĩ xin giải đáp như sau: Chóng mặt có rất ...,143,184


In [8]:
data = data.drop_duplicates(subset=['question_cleaned', 'answer_cleaned'], keep='first')

In [9]:
# Chỉ giữ lại những câu trả lời có độ dài vừa phải để tối ưu tài nguyên
data_filtered = data[(data['a_word_count'] > 80) & (data['a_word_count'] < 1500)].copy()

print(f"Số lượng mẫu còn lại: {len(data_filtered)}")

Số lượng mẫu còn lại: 46465


## Bước 7: Tokenization & Định dạng dữ liệu

### 1. Khởi tạo Tokenizer từ Qwen2-7B-Instruct
- Tokenizer chuyển đổi text thành token IDs
- Qwen2 hỗ trợ tốt Tiếng Việt
- Đặt pad_token = eos_token nếu mô hình chưa có

### 2. Định dạng Prompt - Chat Template
Cấu trúc:
- <|im_start|>system: Định nghĩa vai trò (bác sĩ)
- <|im_start|>user: Câu hỏi từ người dùng
- <|im_start|>assistant: Câu trả lời từ mô hình

### 3. Hàm process_and_tokenize
- Nhận batch (lô) câu hỏi-trả lời
- Tạo prompt format cho toàn bộ lô
- Tokenize batch ngay lập tức
- Xóa cột chữ để tiết kiệm bộ nhớ

### 4. Tham số quan trọng
- max_length=2048: Độ dài tối đa (75% dữ liệu < 2048 tokens)
- batch_size=1000: Xử lý 1000 mẫu mỗi lô 
- num_proc=8: Dùng 8 CPU cores (điều chỉnh theo máy)
- padding=False: Không padding (sẽ làm lúc training)

In [ ]:


# 1. Khởi tạo Tokenizer (Ví dụ với Qwen2 hoặc Llama3 hỗ trợ Tiếng Việt tốt)
model_id = "Qwen/Qwen2-7B-Instruct" 
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Đảm bảo có padding token (một số mô hình như Llama/Qwen cần định nghĩa lại)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Tiền lọc dữ liệu dựa trên thống kê của bạn
# Loại bỏ các câu quá dài (>2000 từ) để tránh cháy RAM GPU và câu quá ngắn
df_filtered = data[(data['a_word_count'] > 50) & (data['a_word_count'] < 2000)].copy()

# 3. Định dạng dữ liệu theo cấu trúc Prompt (Chat Template)
# def format_instruction(example):
#     # Sử dụng format chuẩn của mô hình Instruct
#     prompt = f"<|im_start|>system\nBạn là một bác sĩ chuyên khoa giỏi. Hãy trả lời câu hỏi y tế sau một cách tận tâm.<|im_end|>\n"
#     prompt += f"<|im_start|>user\n{example['question_cleaned']}<|im_end|>\n"
#     prompt += f"<|im_start|>assistant\n{example['answer_cleaned']}<|im_end|>"
#     return {"text": prompt}

# Chuyển từ Pandas sang Hugging Face Dataset và áp dụng format
dataset = Dataset.from_pandas(df_filtered)
# dataset = dataset.map(format_instruction)

# # 4. Hàm Tokenization chính
# def tokenize_function(examples, tok):
#     # max_length chọn 2048 vì dữ liệu của bạn có 75% lượng từ dưới mức này
#     return tok(
#         examples["text"],
#         truncation=True,
#         max_length=2048, 
#         padding=False, # Không padding ở đây để tiết kiệm ổ cứng, sẽ padding theo batch khi train
#         return_tensors=None,
#     )


# 5. Chạy Tokenize
# tokenized_dataset = dataset.map(
#     tokenize_function,
#     batched=True,
#     remove_columns=dataset.column_names, # Xóa các cột chữ thừa để nhẹ dataset
#     #num_proc=1 # Chạy single process để tránh lỗi tokenizer not defined với multiprocessing
#     fn_kwargs={"tok": tokenizer},# Chạy đa nhân để nhanh hơn
#     num_proc=8 # Chạy đa nhân để nhanh hơn
# )

# Tạo hàm: Nhận vào một lô (batch) và xử lý toàn bộ
def process_and_tokenize(examples, tok):
    # 1. Tạo cấu trúc Prompt cho cả lô
    prompts = []
    for q, a in zip(examples['question_cleaned'], examples['answer_cleaned']):
        prompt = f"<|im_start|>system\nBạn là một bác sĩ chuyên khoa giỏi. Hãy trả lời câu hỏi y tế sau một cách tận tâm.<|im_end|>\n"
        prompt += f"<|im_start|>user\n{q}<|im_end|>\n"
        prompt += f"<|im_start|>assistant\n{a}<|im_end|>"
        prompts.append(prompt)
    
    # 2. Tokenize ngay lập tức lô văn bản đó
    return tok(
        prompts,
        truncation=True,
        max_length=2048, 
        padding=False,
        return_tensors=None,
    )

# Khởi tạo Dataset từ Pandas
dataset = Dataset.from_pandas(df_filtered)

print("Đang xử lý và băm dữ liệu ...")
# Chạy 1 lần map duy nhất với cấu hình max tốc độ
tokenized_dataset = dataset.map(
    process_and_tokenize,
    batched=True,
    batch_size=1500, # Tăng số lượng mẫu mỗi lô (Mặc định là 1000)
    remove_columns=dataset.column_names, 
    fn_kwargs={"tok": tokenizer},
    num_proc=8 # Thay đổi số này tùy theo số nhân CPU thực tế của máy
)

# 6. Kiểm tra thử một mẫu sau khi tokenize
print(f"Ví dụ ID của mẫu đầu tiên: {tokenized_dataset[0]['input_ids'][:10]}...")
print(f"Độ dài mẫu này: {len(tokenized_dataset[0]['input_ids'])} tokens")

# 7. Lưu lại
tokenized_dataset.save_to_disk("medical_tokenized_data")

Đang xử lý và băm dữ liệu ...


Map (num_proc=8):   0%|          | 0/62669 [00:00<?, ? examples/s]

Ví dụ ID của mẫu đầu tiên: [151644, 8948, 198, 94917, 37915, 128249, 130103, 128916, 128744, 72067]...
Độ dài mẫu này: 1669 tokens


Saving the dataset (0/1 shards):   0%|          | 0/62669 [00:00<?, ? examples/s]

## Bước 8: Kiểm tra kết quả Tokenization

Mục đích:
- Load lại dữ liệu đã tokenize từ disk
- Decode token IDs ngược lại thành text (xác minh quá trình)
- Đảm bảo dữ liệu được xử lý đúng
- Xem chi tiết cấu trúc prompt sau tokenization

In [11]:
tokenized_dataset = Dataset.load_from_disk("medical_tokenized_data")
sample = tokenized_dataset[0]["input_ids"]

print(f"Độ dài mẫu: {len(sample)} tokens")
print("-" * 50)
print("NỘI DUNG SAU KHI DECODE:")
print("-" * 50)
# Decode từ ID ngược lại thành chữ
print(tokenizer.decode(sample))

Độ dài mẫu: 1669 tokens
--------------------------------------------------
NỘI DUNG SAU KHI DECODE:
--------------------------------------------------
<|im_start|>system
Bạn là một bác sĩ chuyên khoa giỏi. Hãy trả lời câu hỏi y tế sau một cách tận tâm.<|im_end|>
<|im_start|>user
Căng cơ đùi: Nguyên nhân và các phương pháp điều trị<|im_end|>
<|im_start|>assistant
Căng cơ đùi có thể coi là một trong những chấn thương rất hay gặp, đặc biệt là ở những người bệnh thường xuyên chơi thể thao. Khi căng cơ ở đùi, bệnh nhân có thể gặp cảm giác đau đớn, khó chịu và khả năng vận động của bệnh nhân cũng bị ảnh hưởng, từ đó trực tiếp làm giảm chất lượng cuộc sống hàng ngày. Bài viết này được viết dưới sự hướng dẫn chuyên môn của các bác sĩ thuộc khoa Chấn thương chỉnh hình & Y học thể thao Bệnh viện Đa khoa Quốc tế Vinmec. 1. Căng cơ đùi là như thế nào? Cơ đùi gồm ba nhóm chính: Cơ tứ đầu đùi (Qua d ri c e p s): Mặt trước đùi, giúp duỗi gối. Cơ gân kheo (Ham s t rin g): Mặt sau đùi, giúp gập gối. Cơ

## Bước 9: Phân tích thống kê độ dài Token

Thông tin quan trọng:
- Min: Độ dài ngắn nhất (tokens ít nhất)
- Max: Độ dài dài nhất (sát ngưỡng 2048)
- Mean: Độ dài trung bình

Dữ liệu này giúp tối ưu:
- Kích thước batch khi training
- Gradient accumulation steps
- Dự phòng memory GPU

In [12]:
token_lengths = [len(x['input_ids']) for x in tokenized_dataset]
print(f"Độ dài token - Min: {min(token_lengths)}, Max: {max(token_lengths)}, Mean: {np.mean(token_lengths):.0f}")

Độ dài token - Min: 119, Max: 2048, Mean: 1347


## Bước 10: Chia tập dữ liệu & Lưu kết quả cuối cùng

### Tại sao không chia tập test?
Thay vì dùng tập Test để tính độ chính xác (Accuracy) như các bài toán phân loại thông thường, người ta thường đánh giá LLM bằng cách cho con người chat thử trực tiếp.

### Kết quả cuối cùng:
- medical_data_train: Dataset huấn luyện
- medical_data_val: Dataset kiểm tra
- medical_tokenizer: Tokenizer đã lưu (để dùng lúc inference)

In [13]:
# ==========================================
# 8. CHIA TẬP DỮ LIỆU (TRAIN / VALIDATION SPLIT)
# ==========================================
print("Đang chia tập dữ liệu Train và Validation...")

# Tách 5% dữ liệu ngẫu nhiên để làm bài "thi thử" (Validation)
# 95% còn lại để mô hình "học" (Train)
# seed=42 giúp đảm bảo nếu bạn chạy lại code, máy vẫn chia ngẫu nhiên đúng những câu đó
dataset_split = tokenized_dataset.train_test_split(test_size=0.05, seed=42)

train_data = dataset_split['train']
val_data = dataset_split['test']

print(f"Số lượng mẫu dùng để Train: {len(train_data)} câu")
print(f"Số lượng mẫu dùng để Validation: {len(val_data)} câu")

# ==========================================
# 9. LƯU THÀNH PHẨM CUỐI CÙNG
# ==========================================
# Lưu riêng 2 tập này ra ổ cứng
train_data.save_to_disk("medical_data_train")
val_data.save_to_disk("medical_data_val")
tokenizer.save_pretrained("medical_tokenizer")


Đang chia tập dữ liệu Train và Validation...
Số lượng mẫu dùng để Train: 59535 câu
Số lượng mẫu dùng để Validation: 3134 câu


Saving the dataset (0/1 shards):   0%|          | 0/59535 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3134 [00:00<?, ? examples/s]

('medical_tokenizer\\tokenizer_config.json',
 'medical_tokenizer\\chat_template.jinja',
 'medical_tokenizer\\tokenizer.json')